# Conversão de `main.py` para Jupyter Notebook

Este notebook é uma cópia de `main.py`, organizada em **células separadas** para permitir que cada trecho de código seja executado isoladamente.


## 1) Imports e configurações iniciais

Importe as dependências necessárias (equivalente ao topo de `main.py`).


In [1]:
from utils import encontrar_pdfs, regra_divisao
from leitor_pdf import dividir_pdf
import pdfplumber, json

print("Imports carregados")


Imports carregados


## 2) Listar PDFs e iterar sobre eles

Este bloco corresponde ao loop principal de `main.py` que percorre os arquivos na pasta `arquivos/pastateste`.


In [2]:
pdf_folder = "arquivos/pastateste"

pdfs_encontrados = [f for f in encontrar_pdfs(pdf_folder) if f.lower().endswith(".pdf")]
print(f"Encontrados {len(pdfs_encontrados)} PDFs em {pdf_folder}")
for arquivo in pdfs_encontrados:
    print("-", arquivo)


Encontrados 1 PDFs em arquivos/pastateste
- arquivos/pastateste/Tormenta20-Edicao-Jogo-do-Ano-v1.3.pdf


## 3) Processar cada PDF (célula principal)

O bloco a seguir corresponde ao corpo do `for` em `main.py`. Ele abre cada PDF, define classes auxiliares e realiza a extração.


In [3]:
# Escolha um PDF para analisar (altere o índice conforme necessário)
idx = 0
arquivo = pdfs_encontrados[idx]
print(f"Analisando: {arquivo}")


Analisando: arquivos/pastateste/Tormenta20-Edicao-Jogo-do-Ano-v1.3.pdf


In [4]:
# Definições de classes para metadados de texto (originais em `main.py`)

class Topico:
    def __init__(self, texto, tamanho_fonte=None, cor_fonte=None):
        self.texto = texto
        self.tamanho_fonte = tamanho_fonte
        self.cor_fonte = cor_fonte if cor_fonte is None else cor_fonte

        # Garantir que as cores sejam arrays de 3 posições [R, G, B]
        if self.cor_fonte is not None and not isinstance(self.cor_fonte, list):
            self.cor_fonte = [0, 0, 0]


class Titulo:
    def __init__(self, texto, tamanho_fonte=None, cor_fonte=None):
        self.texto = texto
        self.tamanho_fonte = tamanho_fonte
        self.cor_fonte = cor_fonte if cor_fonte is None else cor_fonte

        # Garantir que as cores sejam arrays de 3 posições [R, G, B]
        if self.cor_fonte is not None and not isinstance(self.cor_fonte, list):
            self.cor_fonte = [0, 0, 0]


class Subtitulo:
    def __init__(self, texto, tamanho_fonte=None, cor_fonte=None):
        self.texto = texto
        self.tamanho_fonte = tamanho_fonte
        self.cor_fonte = cor_fonte if cor_fonte is None else cor_fonte

        # Garantir que as cores sejam arrays de 3 posições [R, G, B]
        if self.cor_fonte is not None and not isinstance(self.cor_fonte, list):
            self.cor_fonte = [0, 0, 0]


class Tabela:
    def __init__(self, texto, tamanho_fonte=None, cor_fonte=None):
        self.texto = texto
        self.tamanho_fonte = tamanho_fonte
        self.cor_fonte = cor_fonte if cor_fonte is None else cor_fonte

        # Garantir que as cores sejam arrays de 3 posições [R, G, B]
        if self.cor_fonte is not None and not isinstance(self.cor_fonte, list):
            self.cor_fonte = [0, 0, 0]


In [ ]:
# Seleção de página e recorte das colunas

# Ajuste esse índice para a página desejada (0-index)
pagina_idx = 23

with pdfplumber.open(arquivo) as pdf:
    page = pdf.pages[pagina_idx]
    largura = page.width
    altura = page.height - 32  # cortar rodapé

    coluna_esquerda = page.crop((0, 0, largura / 2, altura))
    coluna_direita = page.crop((largura / 2, 0, largura, altura))


    # Exibe a página como imagem (útil para inspeção rápida)
    from IPython.display import display
    page = page.crop((0, 0, largura, altura))
    im = page.to_image(resolution=200)
    elementos_vermelhos = [
            obj for obj in page.rects + page.curves 
            if obj.get("non_stroking_color") > (0.7, 0.16, 0.16) and obj.get("non_stroking_color") < (0.8, 0.2, 0.2) # Exemplo de RGB normalizado
            or obj.get("stroking_color") == (0.513, 0.202, 0.129)
        ]
    
    red_rect = []
    # rect = {
    #         'x0': 0, 
    #         'y0': 0, 
    #         'x1': 0,
    #         'y1': 0
    #         }
    for i, elem_ver in enumerate(elementos_vermelhos):
        if not (int(elem_ver['y1']) >= int(elementos_vermelhos[i-1]['y1'] - 5) and (int(elem_ver['y1']) <= int(elementos_vermelhos[i-1]['y1'] + 5))):
            red_rect.append(elem_ver)

    for elem_ver in elementos_vermelhos:
        for i, rect in enumerate(red_rect):
            if (int(elem_ver['y1']) >= int(rect['y1'] - 5) and (int(elem_ver['y1']) <= int(rect['y1'] + 5))):
                red_rect[i]['x0'] = int(elem_ver['x0']) if int(elem_ver['x0']) < int(rect['x0']) else int(rect['x0'])
                red_rect[i]['y0'] = int(elem_ver['y0']) if int(elem_ver['y0']) < int(rect['y0']) else int(rect['y0'])
                red_rect[i]['x1'] = int(elem_ver['x1']) if int(elem_ver['x1']) > int(rect['x1']) else int(rect['x1'])
                red_rect[i]['y1'] = int(elem_ver['y1']) if int(elem_ver['y1']) > int(rect['y1']) else int(rect['y1'])
                


    if not elementos_vermelhos:
        print("Nenhuma linha decorativa encontrada.")
    # else:
    #     for elem_ver in elementos_vermelhos:
    #         print(elem_ver)

    # Desenha retângulos sobre os elementos vermelhos e exibe a imagem
    if elementos_vermelhos:
        print(len(red_rect))
        print(len(elementos_vermelhos))
        
        # elem0 = red_rect[1]
        # for rr in red_rect:
        #     print(int(rr['y0']), int(rr['y1']))
        # ponto_inicio = (elem0["x0"], elem0.get("top", elem0.get("y0")))
        # im.draw_circle(ponto_inicio, stroke="green", fill=None, radius=5)
        # display(im)
    # display(im.original)
    # display(im.debug_tablefinder(table_settings={"horizontal_strategy": "lines_strict"}))

    tables = page.find_tables(table_settings={})
    if tables:
        tbl = tables[-1]
        x0, top, x1, bottom = tbl.bbox
        cropped = page.crop((x0, top, x1, bottom))
        # display(cropped.to_image(resolution=200).original)
    else:
        print("Nenhuma tabela encontrada na página")

    pagina_completa = coluna_esquerda.extract_text_lines() + coluna_direita.extract_text_lines()


2
5


In [21]:
# Iterar sobre as linhas extraídas e detectar padrões

for linha in pagina_completa:
    R, G, B = (
        linha["chars"][0]["non_stroking_color"][0],
        linha["chars"][0]["non_stroking_color"][1],
        linha["chars"][0]["non_stroking_color"][2],
    )
    tam_fonte = linha["chars"][0]["size"]

    params_title = all(pixel_color * 255 > 200 for pixel_color in (R, G, B)) and tam_fonte >= 15

    if "Tabela" in linha["text"]:
        list_tabela = Tabela(linha["text"], tam_fonte, [R, G, B])
        print(linha["chars"][0])

    # if (params_title):
    #     print(linha['text'])
    #     print(tam_fonte)


{'matrix': (12.0, 0.0, 0.0, 12.0, 344.724, 380.4483), 'fontname': 'ITPJJY+IowanOldStyle-Bold', 'adv': 0.6890000000000001, 'upright': True, 'x0': 344.724, 'y0': 374.97630000000004, 'x1': 352.99199999999996, 'y1': 386.97630000000004, 'width': 8.267999999999972, 'height': 12.0, 'size': 12.0, 'mcid': None, 'tag': None, 'object_type': 'char', 'page_number': 24, 'ncs': 'ICCBased', 'text': 'T', 'stroking_color': (0,), 'non_stroking_color': (0.513, 0.202, 0.129), 'top': 392.5517, 'bottom': 404.5517, 'doctop': 18321.6957}
